In [ ]:
"""eod_sale_product — Bronze ingest.
 
All relational data now comes from PostgreSQL (7reward/som/oms) — including the
three dims (product_uom_upc, dict_stores, purchase_price_timeline) that used to be
pulled from ClickHouse. This takes ClickHouse off the ingest critical path.
 
  PRODUCTION: PostgreSQL arrives via OneLake shortcut, so every relational table
    (the 9 operational tables + the 3 dims) already exists as bronze.* — there is
    NO ingest code for them here. This function only handles the sources that
    genuinely need code: Mongo (daily window) and the DLM REST API.
 
  LOCAL / DEV multi-source extract: set a `pg-jdbc` secret (JDBC URL to a Postgres
    holding all 12 tables) and, optionally, a `dlm-url` secret pointing at the mock
    API service. bronze_ingest then extracts every source over its real connector
    (Mongo connector, JDBC, requests) instead of relying on the shortcut.
    See pipelines/eod_sale_product/{postgres_seed.sql, mongo_seed.js, api_service.py}.
"""
import json
 
import requests
 
from ssv_data.io.readers import Readers
from ssv_data.io.writers import write_delta

# PostgreSQL extraction policy (JDBC dev path; prod arrives via shortcut).
# Windowed transactional tables: created_at is write-once, and rows are re-read with
# their CURRENT values — late UPDATEs (status, rating, comment) are still captured on
# backfill; only rows INSERTed with an out-of-cover created_at are missed.
PG_WINDOWED = {
    "delivery_orders": "created_at",
    "sale_transactions": "created_at",
    "point_histories": "created_at",
}
# Children of delivery_orders: details has no timestamp; audit events can land after
# midnight (order created 23:30 completes 00:30 next day) -> semi-join on the
# windowed parent instead of cutting by the event's own time.
PG_ORDER_CHILDREN = ["delivery_order_audits", "delivery_order_details"]
# Master data: small, no timestamps -> full extract every run.
PG_FULL = ["users", "oms_product", "oms_store", "mapping_payment_method"]
# Dims migrated off ClickHouse -> maintained in Postgres too, so they arrive the
# same way as the other relational tables.
DIM_TABLES = ["product_uom_upc", "dict_stores", "purchase_price_timeline"]
DLM_URL_DEFAULT = "https://dlm.sevensystem.vn/api/hq/dlm/partner_sources"

In [ ]:
def _opt(ctx, key):
    """Optional secret: None if no resolver / key is configured."""
    try:
        return ctx.secret(key)
    except Exception:
        return None
 
 
def _mongo_window(w) -> str:
    """$match on documentDate for the run's UTC window (extended-JSON dates)."""
    pipeline = [{"$match": {"documentDate": {
        "$gte": {"$date": w.utc_lo.isoformat() + "Z"},
        "$lt": {"$date": w.utc_hi.isoformat() + "Z"},
    }}}]
    return json.dumps(pipeline)
 
 
def _read_dlm_partner_sources(ctx):
    url = _opt(ctx, "dlm-url") or DLM_URL_DEFAULT
    resp = requests.get(
        url,
        headers={"Content-Type": "application/json",
                 "Authorization": ctx.secret("dlm-auth")},
        timeout=20,
    )
    resp.raise_for_status()
    return ctx.spark.createDataFrame(resp.json())

In [ ]:
# ---- per-source entrypoints (each writes ONLY its bronze tables) ----
def ingest_mongo(ctx) -> None:
    """MongoDB: bills (windowed on documentDate) + partner_store (full). Nested JSON intact."""
    r = Readers(ctx)
    pl = _mongo_window(ctx.window)
    write_delta(ctx, r.mongo("report", "sale_bill", pl), ctx.bronze("sale_bill"))
    write_delta(ctx, r.mongo("report", "sale_return_bill", pl), ctx.bronze("sale_return_bill"))
    write_delta(ctx, r.mongo("delivery_order", "partner_store"), ctx.bronze("partner_store"))
    ctx.logger.info("bronze: Mongo ingested (sale_bill, sale_return_bill, partner_store)")
 
 
def ingest_dlm(ctx) -> None:
    """DLM REST API -> partner_sources (URL overridable with a dlm-url secret)."""
    write_delta(ctx, _read_dlm_partner_sources(ctx), ctx.bronze("partner_sources"))
    ctx.logger.info("bronze: DLM partner_sources ingested")
 
 
def ingest_postgres(ctx, full_load: bool = False) -> None:
    """PostgreSQL over JDBC (needs pg-jdbc): windowed transactional tables, semi-joined
    order children, full masters + dims. Predicates are pushed down to Postgres, so
    bytes transferred scale with the day's volume, not table size.
 
    full_load=True falls back to `SELECT *` for every table — the exact repair path
    for a backfill that must also capture rows INSERTed outside the cover window.
    """
    r = Readers(ctx)
    if full_load:
        for t in list(PG_WINDOWED) + PG_ORDER_CHILDREN + PG_FULL + DIM_TABLES:
            write_delta(ctx, r.jdbc("pg_jdbc", f"SELECT * FROM public.{t}"), ctx.bronze(t))
        ctx.logger.info("bronze: PG + dims ingested via JDBC (FULL LOAD)")
        return
    for t, ts in PG_WINDOWED.items():
        write_delta(ctx, r.jdbc_windowed("pg_jdbc", f"public.{t}", ts), ctx.bronze(t))
    for t in PG_ORDER_CHILDREN:
        write_delta(ctx, r.jdbc_parent_windowed("pg_jdbc", f"public.{t}",
                                                "public.delivery_orders", "order_id", "created_at"),
                    ctx.bronze(t))
    for t in PG_FULL + DIM_TABLES:
        write_delta(ctx, r.jdbc("pg_jdbc", f"SELECT * FROM public.{t}"), ctx.bronze(t))
    ctx.logger.info("bronze: PG ingested via JDBC (windowed txn + semi-joined children + full masters/dims)")

In [ ]:
# ---- combined ingest (single-notebook path); each source gated by its secret ----
def bronze_ingest(ctx) -> None:
    if _opt(ctx, "mongo-conn"):
        ingest_mongo(ctx)
    else:
        ctx.logger.info("bronze: Mongo SKIPPED (no mongo-conn) — expected already in bronze "
                        "(e.g. a Copy data activity)")
 
    if _opt(ctx, "dlm-auth"):
        ingest_dlm(ctx)
    else:
        ctx.logger.info("bronze: DLM SKIPPED (no dlm-auth) — expected already in bronze")
 
    if _opt(ctx, "pg-jdbc"):
        ingest_postgres(ctx)
    else:
        ctx.logger.info("bronze: PG + dims via OneLake shortcut / external (no pg-jdbc)")